**Autor**: Juan Alejandro Carrillo Jaimes

**Creación**: 05-08-2026

**Ult Modificación**: 09-08-2026

**Materia**: Mét. Apren Auto para Toma de Decisiones 20262  - A

**Descripción**: Script de aprendizaje supervisado sobre el dataset Breast Cancer
Wisconsin (Original). Se entrena un modelo de regresión logística para
clasificar tumores como benignos o malignos, dividiendo los datos en
conjuntos de entrenamiento y prueba, y se reporta el accuracy obtenido.

In [1]:
!pip3 install -U ucimlrepo

In [31]:
# Carga de librerías

import pandas as pd

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

# Semilla para reproducibilidad
RANDOM_STATE = 42

### **1. Carga del conjunto de datos**

Obtener el dataset directamente desde UCI y dejarlo disponible como dataframes de pandas, listos para trabajar.

In [3]:
def impute_missing_values(df: pd.DataFrame) -> pd.DataFrame:
    """Completa los datos faltantes de atributos numéricos usando la media.

    Args:
        df (pd.DataFrame): Conjunto de datos con posibles faltantes.

    Returns:
        pd.DataFrame: Copia del dataframe con los faltantes imputados.
    """
    df_imputed = df.copy()

    for col in df_imputed.columns:
        if df_imputed[col].isnull().any():
            df_imputed[col] = df_imputed[col].fillna(df_imputed[col].mean())

    return df_imputed

In [12]:
# Fetch dataset
breast_cancer_wisconsin_original = fetch_ucirepo(id=15)

# Data (as pandas dataframes)
X = breast_cancer_wisconsin_original.data.features
y = breast_cancer_wisconsin_original.data.targets
print(f'Shape: {X.shape}')
print("--------------------------")
print(X.isnull().sum())

Shape: (699, 9)
--------------------------
Clump_thickness                 0
Uniformity_of_cell_size         0
Uniformity_of_cell_shape        0
Marginal_adhesion               0
Single_epithelial_cell_size     0
Bare_nuclei                    16
Bland_chromatin                 0
Normal_nucleoli                 0
Mitoses                         0
dtype: int64


In [14]:
# El atributo Bare_nuclei tiene valores faltantes en el dataset original;
# se imputan con la media, ya que LogisticRegression no acepta NaN.
X_clean = impute_missing_values(X)
print(f"Valores faltantes por atributo:\n{X_clean.isnull().sum()}")

Valores faltantes por atributo:
Clump_thickness                0
Uniformity_of_cell_size        0
Uniformity_of_cell_shape       0
Marginal_adhesion              0
Single_epithelial_cell_size    0
Bare_nuclei                    0
Bland_chromatin                0
Normal_nucleoli                0
Mitoses                        0
dtype: int64


### **2. División en entrenamiento y prueba**

Separar los datos para poder entrenar el modelo con una parte y evaluarlo con datos que no vio durante el entrenamiento.

In [15]:
def split_data(X: pd.DataFrame, y: pd.DataFrame, test_size: float = 0.3):
    """Divide el conjunto de datos en entrenamiento y prueba.

    Args:
        X (pd.DataFrame): Atributos de entrada.
        y (pd.DataFrame): Atributo objetivo (clase).
        test_size (float): Proporción del conjunto de prueba.

    Returns:
        tuple: X_train, X_test, y_train, y_test.
    """
    return train_test_split(X, y, test_size=test_size, random_state=RANDOM_STATE, stratify=y)

In [16]:
# Se usa una proporción estándar de 70% entrenamiento / 30% prueba
X_train, X_test, y_train, y_test = split_data(X_clean, y, test_size=0.3)

print(f"Registros de entrenamiento: {X_train.shape[0]}")
print(f"Registros de prueba: {X_test.shape[0]}")

Registros de entrenamiento: 489
Registros de prueba: 210


### **3. Construcción del modelo (regresión logística)**

Entrenar por regresión logística usando únicamente el conjunto de entrenamiento. (70%)

In [17]:
def train_model(model, X_train, y_train):
    """Entrena un modelo de clasificación con el conjunto de entrenamiento.

    Args:
        model: Instancia de un clasificador de scikit-learn.
        X_train (pd.DataFrame): Atributos de entrenamiento.
        y_train (pd.DataFrame): Clase de entrenamiento.

    Returns:
        Modelo entrenado.
    """
    model.fit(X_train, y_train.values.ravel())
    return model

In [18]:
log_model = train_model(LogisticRegression(random_state=RANDOM_STATE, max_iter=1000), X_train, y_train)

## Evaluación

In [24]:
def get_confusion_matrix(model, X_test: pd.DataFrame, y_test: pd.DataFrame) -> pd.DataFrame:
    """Calcula la matriz de confusión del modelo sobre el conjunto de prueba.

    Args:
        model: Modelo de clasificación ya entrenado.
        X_test (pd.DataFrame): Atributos de prueba.
        y_test (pd.DataFrame): Clase real de prueba.

    Returns:
        pd.DataFrame: Matriz de confusión con etiquetas legibles.
    """
    y_pred = model.predict(X_test)
    labels = sorted(y_test.iloc[:, 0].unique())  # [2, 4] -> [benigno, maligno]
    label_names = {2: "Benigno (2)", 4: "Maligno (4)"}

    matrix = confusion_matrix(y_test, y_pred, labels=labels)
    matrix_df = pd.DataFrame(
        matrix,
        index=[f"Real: {label_names[l]}" for l in labels],
        columns=[f"Predicho: {label_names[l]}" for l in labels],
    )
    return matrix_df

In [25]:
def predict_single_case(model, feature_names: list, values: list) -> int:
    """Predice la clase de un único registro definido manualmente.

    Args:
        model: Modelo de clasificación ya entrenado.
        feature_names (list): Nombres de los atributos, en el mismo orden
            usado durante el entrenamiento.
        values (list): Valores del registro a predecir, en el mismo orden
            que feature_names.

    Returns:
        int: Clase predicha (2 = benigno, 4 = maligno).
    """
    sample = pd.DataFrame([values], columns=feature_names)
    prediction = model.predict(sample)[0]

    label_names = {2: "Benigno", 4: "Maligno"}
    print(f"Predicción: {prediction} ({label_names[prediction]})")

    return prediction

In [32]:
def evaluate_model(model, X_test: pd.DataFrame, y_test: pd.DataFrame) -> dict:
    """Evalúa un modelo entrenado calculando métricas de desempeño
    sobre el conjunto de prueba.

    Args:
        model: Modelo de clasificación ya entrenado.
        X_test (pd.DataFrame): Atributos de prueba.
        y_test (pd.DataFrame): Clase real de prueba.

    Returns:
        dict: Diccionario con accuracy, precision, recall y f1_score.
    """
    y_pred = model.predict(X_test)

    return {
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, pos_label=4),
        "recall": recall_score(y_test, y_pred, pos_label=4),
        "f1_score": f1_score(y_test, y_pred, pos_label=4),
    }

In [33]:
metrics = evaluate_model(log_model, X_test, y_test)

for metric_name, value in metrics.items():
    print(f"{metric_name}: {value:.4f}")

accuracy: 0.9571
precision: 0.9315
recall: 0.9444
f1_score: 0.9379


In [34]:
conf_matrix = get_confusion_matrix(log_model, X_test, y_test)
print("Matriz de confusión:")
conf_matrix

Matriz de confusión:


,Predicho: Benigno (2),Predicho: Maligno (4)
Real: Benigno (2),133,5
Real: Maligno (4),4,68


In [35]:
# Caso claramente benigno (valores bajos en todos los atributos)
benign_example = [2, 1, 1, 1, 2, 1, 2, 1, 1]
predict_single_case(log_model, list(X.columns), benign_example)

# Caso claramente maligno (valores altos en todos los atributos)
malignant_example = [8, 9, 8, 7, 6, 9, 8, 7, 3]
predict_single_case(log_model, list(X.columns), malignant_example)

Predicción: 2 (Benigno)
Predicción: 4 (Maligno)


np.int64(4)